In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [ ]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [ ]:
MODEL_PATH = "/kaggle/input/datasets/venkat23f1000054/deberta-small"
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)
model.to(device)
model.eval()

In [ ]:
def get_score(prompt, option):
    inputs = tokenizer(
        prompt,
        option,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )
    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
    }
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(
            outputs.logits,
            dim=1
        )
        score = probs[:,1].item()
    return score

In [ ]:
options = ["A","B","C","D","E"]
predictions = []
for _, row in test.iterrows():
    scores = []
    for option in options:
        score = get_score(
            row["prompt"],
            str(row[option])
        )
        scores.append(score)
    ranked = np.argsort(scores)[::-1]
    top3 = [options[i] for i in ranked[:3]]
    predictions.append(
        " ".join(top3)
    )

In [ ]:
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [ ]:
sample["Prediction"] = predictions
sample.head()

In [ ]:
sample.to_csv("submission1.csv", index=False)